<a href="https://colab.research.google.com/github/tazir-shaif/ai-engineering-portfolio/blob/main/module-2-data-pipelines/Module_2_Session_1_CSV_Pipelines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 2 — Session 1: CSV Data Pipelines

## What we are building
Swiggy receives thousands of orders every day — all stored as structured data (CSV files).
Before we can use an LLM to analyze this data, we need a **data pipeline** that:
1. Loads the CSV
2. Cleans it
3. Converts rows into natural language that an LLM can understand

## Real-world equivalent
At Swiggy, this would run on AWS Glue (ETL) feeding into SageMaker or Bedrock.

## Step 1 — Install and Import Libraries
We need two libraries:
- `pandas` — to load and manipulate CSV data (you already know this from Module 1)
- `groq` — to call the LLM (you already know this too)

No new libraries this session — we practice with what we know.

In [2]:
# Install required libraries — needed every time we open a new Colab notebook
!pip install groq langchain langchain-groq -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 6.0 MB/s eta 0:00:00


In [3]:
# We already installed these in Module 0 — just importing them here
import pandas as pd  # pandas helps us work with tables/CSV files
from groq import Groq  # Groq gives us access to the LLM
import os  # os helps us read environment variables like API keys
from google.colab import userdata  # lets us safely read Colab Secrets

## Step 2 — Create Sample Swiggy Orders CSV
In real life, Swiggy has millions of order rows in S3.
For practice, we create a small CSV with 10 orders right inside Colab.

In [4]:
# Creating a sample Swiggy orders dataset
data = {
    "order_id": [101, 102, 103, 104, 105, 106, 107, 108, 109, 110],
    "customer_name": ["Aarav", "Priya", "Rohit", "Sneha", "Kiran",
                      "Meera", "Arjun", "Divya", "Raj", "Anita"],
    "restaurant": ["McDonald's", "Domino's", "Biryani Blues", "McDonald's",
                   "Domino's", "Biryani Blues", "McDonald's", "Domino's",
                   "Biryani Blues", "McDonald's"],
    "item_ordered": ["Burger", "Pizza", "Biryani", "Fries", "Pasta",
                     "Biryani", "McFlurry", "Garlic Bread", "Biryani", "Burger"],
    "order_value": [250, 450, 380, 120, 320, 400, 150, 180, 390, 260],
    "delivery_time_mins": [32, 45, 28, 55, 38, 22, 60, 40, 25, 48],
    "rating": [4, 3, 5, 2, 4, 5, 1, 3, 5, 4]
}

# Convert the dictionary into a pandas DataFrame (a table)
df = pd.DataFrame(data)

# Save it as a CSV file
df.to_csv("swiggy_orders.csv", index=False)

print("CSV created successfully!")
print(df)

CSV created successfully!
   order_id customer_name     restaurant  item_ordered  order_value  \
0       101         Aarav     McDonald's        Burger          250   
1       102         Priya       Domino's         Pizza          450   
2       103         Rohit  Biryani Blues       Biryani          380   
3       104         Sneha     McDonald's         Fries          120   
4       105         Kiran       Domino's         Pasta          320   
5       106         Meera  Biryani Blues       Biryani          400   
6       107         Arjun     McDonald's      McFlurry          150   
7       108         Divya       Domino's  Garlic Bread          180   
8       109           Raj  Biryani Blues       Biryani          390   
9       110         Anita     McDonald's        Burger          260   

   delivery_time_mins  rating  
0                  32       4  
1                  45       3  
2                  28       5  
3                  55       2  
4                  38       4  


## Step 3 — Load and Explore the CSV
Before feeding data to an LLM, we always explore it first.
We need to understand: shape, column names, any missing values.

In [5]:
# Load the CSV back — simulating how we would read from S3 in real life
df = pd.read_csv("swiggy_orders.csv")

# How many rows and columns?
print("Shape:", df.shape)

# What are the column names?
print("\nColumns:", df.columns.tolist())

# Show first 5 rows
print("\nFirst 5 rows:")
print(df.head())

# Any missing values?
print("\nMissing values:")
print(df.isnull().sum())

Shape: (10, 7)

Columns: ['order_id', 'customer_name', 'restaurant', 'item_ordered', 'order_value', 'delivery_time_mins', 'rating']

First 5 rows:
   order_id customer_name     restaurant item_ordered  order_value  \
0       101         Aarav     McDonald's       Burger          250   
1       102         Priya       Domino's        Pizza          450   
2       103         Rohit  Biryani Blues      Biryani          380   
3       104         Sneha     McDonald's        Fries          120   
4       105         Kiran       Domino's        Pasta          320   

   delivery_time_mins  rating  
0                  32       4  
1                  45       3  
2                  28       5  
3                  55       2  
4                  38       4  

Missing values:
order_id              0
customer_name         0
restaurant            0
item_ordered          0
order_value           0
delivery_time_mins    0
rating                0
dtype: int64


## Step 4 — Filter and Clean the Data
We don't send all 10 rows to the LLM — that wastes tokens and money.
We filter only the problematic orders (rating below 3) and send those for analysis.

In [6]:
# Filter orders with low ratings (less than or equal to 2)
# These are unhappy customers that need attention
low_rated = df[df["rating"] <= 2]

print("Low rated orders:")
print(low_rated)
print(f"\nTotal problematic orders: {len(low_rated)}")

Low rated orders:
   order_id customer_name  restaurant item_ordered  order_value  \
3       104         Sneha  McDonald's        Fries          120   
6       107         Arjun  McDonald's     McFlurry          150   

   delivery_time_mins  rating  
3                  55       2  
6                  60       1  

Total problematic orders: 2


## Step 5 — Convert Rows to Natural Language
LLMs don't understand tables — they understand text.
We convert each row into a sentence that describes the order naturally.
This is called "serialization" — turning structured data into natural language.

In [7]:
# Convert each low-rated order row into a natural language sentence
# This is what we will actually send to the LLM
def row_to_text(row):
    return (
        f"Order ID {row['order_id']}: Customer {row['customer_name']} ordered "
        f"{row['item_ordered']} from {row['restaurant']}. "
        f"Order value was ₹{row['order_value']}. "
        f"Delivery took {row['delivery_time_mins']} minutes. "
        f"Customer gave a rating of {row['rating']} out of 5."
    )

# Apply this function to every row in our filtered dataframe
low_rated["text"] = low_rated.apply(row_to_text, axis=1)

# Print each converted sentence
for text in low_rated["text"]:
    print(text)
    print()

Order ID 104: Customer Sneha ordered Fries from McDonald's. Order value was ₹120. Delivery took 55 minutes. Customer gave a rating of 2 out of 5.

Order ID 107: Customer Arjun ordered McFlurry from McDonald's. Order value was ₹150. Delivery took 60 minutes. Customer gave a rating of 1 out of 5.



/tmp/ipykernel_834/3814497465.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  low_rated["text"] = low_rated.apply(row_to_text, axis=1)


In [8]:
# .copy() tells pandas this is a new independent dataframe, not a slice
# This removes the SettingWithCopyWarning
low_rated = df[df["rating"] <= 2].copy()

# Convert each low-rated order row into a natural language sentence
def row_to_text(row):
    return (
        f"Order ID {row['order_id']}: Customer {row['customer_name']} ordered "
        f"{row['item_ordered']} from {row['restaurant']}. "
        f"Order value was ₹{row['order_value']}. "
        f"Delivery took {row['delivery_time_mins']} minutes. "
        f"Customer gave a rating of {row['rating']} out of 5."
    )

low_rated["text"] = low_rated.apply(row_to_text, axis=1)

for text in low_rated["text"]:
    print(text)
    print()

Order ID 104: Customer Sneha ordered Fries from McDonald's. Order value was ₹120. Delivery took 55 minutes. Customer gave a rating of 2 out of 5.

Order ID 107: Customer Arjun ordered McFlurry from McDonald's. Order value was ₹150. Delivery took 60 minutes. Customer gave a rating of 1 out of 5.



## Step 6 — Send Data to LLM for Analysis
Now we combine both order texts into one prompt and ask the LLM to:
1. Identify what went wrong
2. Suggest what Swiggy should do next

In [9]:
# Set up Groq client using our saved API key
client = Groq(api_key=userdata.get("GROQ_API_KEY"))

# Combine all low rated order texts into one block
all_orders_text = "\n".join(low_rated["text"].tolist())

# Build the prompt
prompt = f"""You are a customer experience analyst at Swiggy.
Below are orders where customers gave low ratings.
For each order, identify what went wrong and suggest one action Swiggy should take.

Orders:
{all_orders_text}

Give your response in this format for each order:
Order ID:
Problem:
Suggested Action:
"""

# Send to LLM
response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[{"role": "user", "content": prompt}]
)

# Print the LLM response
print(response.choices[0].message.content)

Order ID 104: 
Problem: The delivery time of 55 minutes was significantly higher than expected and may have impacted the customer's experience.
Suggested Action: Swiggy should optimize their route planning algorithm to reduce delivery times, and provide customers with estimated delivery times based on historical data for specific restaurants.

Order ID 107: 
Problem: The customer's experience was not just impacted by delivery time, but also the quality of the order which resulted in a 1-star rating. Considering that McFlurry is typically a frozen dessert that requires quick handling and storage, the delay in delivery may have caused it to spoil or melt during the long journey.
Suggested Action: Swiggy should implement a 'temperature-sensitive' or 'time-critical' order flag to prioritize such orders and communicate with restaurants to adopt measures like refrigeration during transit to ensure that perishable items like McFlurry are delivered within a safe time frame.


## Step 7 — Save LLM Output back to CSV
In production, Swiggy would save these insights to a database or S3.
We save it as a CSV to complete the pipeline loop.

In [10]:
# Save the LLM response as a new column in our dataframe
# For now we store the full response — in next sessions we will parse it properly

llm_response = response.choices[0].message.content

# Add response to our dataframe
low_rated["llm_analysis"] = llm_response

# Save to CSV
low_rated.to_csv("swiggy_low_rated_analysis.csv", index=False)

print("Saved successfully!")
print(low_rated[["order_id", "customer_name", "rating", "llm_analysis"]])

Saved successfully!
   order_id customer_name  rating  \
3       104         Sneha       2   
6       107         Arjun       1   

                                        llm_analysis  
3  Order ID 104: \nProblem: The delivery time of ...  
6  Order ID 104: \nProblem: The delivery time of ...  
